In [ ]:
import pandas as pd
import geopandas as gpd
import requests
import folium
import json
import plotly.express as px
import plotly.graph_objects as go
from rapidfuzz import process, fuzz
import geopandas as gpd
import matplotlib.pyplot as plt
from unidecode import unidecode


In [ ]:
denue = gpd.read_file('/Users/alanrobles/Documents/ALAN/GEOSTATS/CSV descragados (data)/denue_19_shp/conjunto_de_datos/denue_inegi_19_.shp')

In [ ]:
denue.head()

In [ ]:
denue['nombre_act'].unique()

In [ ]:
transparencia = pd.read_excel("/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/InformeTransparencia2024.xlsx")

In [ ]:
transparencia 

In [ ]:
transparencia_nl = transparencia[transparencia['Entidad federativa'] == 'NUEVO LEON'].copy()


In [ ]:
transparencia_nl

In [ ]:
def normalizar(s):
    return str(s).strip().upper()

denue['raz_social_norm'] = denue['raz_social'].apply(normalizar)
transparencia_nl['razon_social_norm'] = transparencia_nl['Razón social'].apply(normalizar)

# --- 2. Match exacto ---
exactos = denue.merge(
    transparencia_nl,
    left_on='raz_social_norm',
    right_on='razon_social_norm',
    how='inner'
)

print(f"Matches exactos: {len(exactos)}")
print(exactos[['raz_social', 'Razón social']].head(20))

In [ ]:
# --- 3. Fuzzy match (solo los que NO tuvieron match exacto) ---
denue_sin_match = denue[~denue['raz_social_norm'].isin(exactos['raz_social_norm'])].copy()

opciones = transparencia_nl['razon_social_norm'].tolist()

def buscar_fuzzy(nombre, opciones, score_min=85):
    resultado = process.extractOne(
        nombre,
        opciones,
        scorer=fuzz.token_sort_ratio  # tolera palabras reordenadas
    )
    if resultado and resultado[1] >= score_min:
        return resultado[0], resultado[1]
    return None, None

denue_sin_match[['match_fuzzy', 'score']] = denue_sin_match['raz_social_norm'].apply(
    lambda x: pd.Series(buscar_fuzzy(x, opciones))
)

fuzzy_matches = denue_sin_match[denue_sin_match['match_fuzzy'].notna()].copy()

# Adjuntar los datos de transparencia al resultado fuzzy
fuzzy_con_datos = fuzzy_matches.merge(
    transparencia_nl,
    left_on='match_fuzzy',
    right_on='razon_social_norm',
    how='left'
)

print(f"Matches fuzzy (score >= 90): {len(fuzzy_con_datos)}")
print(fuzzy_con_datos[['raz_social', 'Razón social', 'score']].sort_values('score', ascending=False).head(20))

In [ ]:
destino = pd.read_excel("/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/InformeTransparencia2024.xlsx", sheet_name='Destino de donativos')
# Filtrar Nuevo León
destino_nl = destino[destino['Entidad federativa'] == 'Nuevo León'].copy()

# Contar registros por municipio
conteo_municipio = destino_nl.groupby('Municipio').size().reset_index(name='conteo')

print(f"Municipios únicos: {conteo_municipio['Municipio'].nunique()}")
print(conteo_municipio.sort_values('conteo', ascending=False).head(10))

In [ ]:
municipios = gpd.read_file("/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/data/2025_1_00_MUN.shp")


municipios_nl = municipios[municipios['CVE_ENT'] == '19'].copy()

amm = [
    'Monterrey', 'Guadalupe', 'San Nicolás de los Garza', 'Apodaca',
    'General Escobedo', 'Santa Catarina', 'García', 'Juárez',
    'San Pedro Garza García', 'Pesquería',
]

municipios_nl = municipios_nl[municipios_nl['NOMGEO'].isin(amm)].copy()


In [ ]:
municipios_nl

In [ ]:

# --- 3. Normalizar quitando acentos y pasando a mayúsculas ---
def normalizar(s):
    return unidecode(str(s)).strip().upper()

municipios_nl['nom_norm'] = municipios_nl['NOMGEO'].apply(normalizar)
conteo_municipio['mun_norm'] = conteo_municipio['Municipio'].apply(normalizar)

# --- 4. Merge ---
municipios_nl = municipios_nl.merge(
    conteo_municipio[['mun_norm', 'conteo']],
    left_on='nom_norm',
    right_on='mun_norm',
    how='left'
)
municipios_nl['conteo'] = municipios_nl['conteo'].fillna(0)

# Verificar qué quedó sin match
print(municipios_nl[['NOMGEO', 'conteo']].sort_values('conteo', ascending=False).to_string())

# --- 5. Graficar ---
fig, ax = plt.subplots(figsize=(14, 12))

municipios_nl.plot(
    ax=ax,
    column='conteo',
    cmap='YlOrRd',
    edgecolor='#555555',
    linewidth=0.5,
    legend=True,
    legend_kwds={'label': 'Número de donativos destinados', 'orientation': 'vertical'}
)

# Etiquetas solo para municipios con datos
con_datos = municipios_nl[municipios_nl['conteo'] > 0]
for _, row in con_datos.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row['NOMGEO']}\n({int(row['conteo'])})",
        xy=(centroid.x, centroid.y),
        fontsize=6,
        ha='center',
        color='#222222'
    )

ax.set_title('Destino de donativos por municipio — Nuevo León', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
destino = pd.read_excel("/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/InformeTransparencia2024.xlsx", sheet_name='Destino de donativos')







In [ ]:
municipios = gpd.read_file("/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/data/2025_1_00_MUN.shp")



In [ ]:
# --- 1. Conteo por municipio ---
conteo = destino.groupby('Municipio').size().reset_index(name='conteo')

# --- 2. Normalizar y hacer merge ---
def norm(s):
    return unidecode(str(s)).strip().upper()

municipios['nom_norm'] = municipios['NOMGEO'].apply(norm)
conteo['mun_norm'] = conteo['Municipio'].apply(norm)

mapa = municipios.to_crs(epsg=4326).merge(conteo, left_on='nom_norm', right_on='mun_norm', how='left')
mapa['conteo'] = mapa['conteo'].fillna(0)

# --- 3. Plotly ---
fig = px.choropleth_mapbox(
    mapa,
    geojson=mapa.__geo_interface__,
    locations=mapa.index,
    color='conteo',
    color_continuous_scale='YlOrRd',
    hover_name='NOMGEO',
    hover_data={'conteo': True},
    mapbox_style='carto-positron',
    zoom=4,
    center={'lat': 23.6, 'lon': -102.5},
    opacity=0.7,
    title='Destino de donativos por municipio — Nacional',
    labels={'conteo': 'Donativos'}
)

fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.grid': True,
    'grid.alpha': 0.15,
    'grid.color': '#000000',
    'axes.axisbelow': True,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

COLORS = {
    'rojo':    '#8B2F2F',
    'coral':   '#C0683A',
    'verde':   '#3B6D11',
    'azul':    '#185FA5',
    'morado':  '#534AB7',
    'ambar':   '#854F0B',
    'teal':    '#0F6E56',
    'gris':    '#73726C',
}

# ── 1. Concentración de monto por entidad federativa ─────────────────────────
top_entidades = (
    destino.groupby('Entidad federativa')['Monto']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_entidades.index, top_entidades.values,
               color=COLORS['rojo'], height=0.6)

for bar, val in zip(bars, top_entidades.values):
    ax.text(val + top_entidades.values.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'${val/1e6:.1f}M', va='center', fontsize=10, color=COLORS['rojo'], fontweight='bold')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
ax.set_xlabel('Monto total donado (MXN)', fontsize=11, color='#555')
ax.set_title('Concentración de monto donado\npor entidad federativa — top 10',
             fontsize=14, fontweight='bold', color='#222', pad=16)
ax.tick_params(axis='y', labelsize=11)
ax.tick_params(axis='x', labelsize=10, colors='#666')
ax.spines['bottom'].set_color('#ddd')
plt.tight_layout()
plt.savefig('g1_monto_entidad.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2. Distribución de montos por rango ──────────────────────────────────────
bins   = [0, 1_000, 10_000, 50_000, 100_000, 500_000, 1_000_000, np.inf]
labels = ['< $1K', '$1K–$10K', '$10K–$50K', '$50K–$100K',
          '$100K–$500K', '$500K–$1M', '> $1M']

destino['rango_monto'] = pd.cut(destino['Monto'], bins=bins, labels=labels)
conteo_rangos = destino['rango_monto'].value_counts().reindex(labels)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(conteo_rangos.index, conteo_rangos.values,
              color=COLORS['coral'], width=0.6)

for bar, val in zip(bars, conteo_rangos.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + conteo_rangos.max() * 0.01,
            f'{int(val):,}', ha='center', fontsize=10, color=COLORS['coral'], fontweight='bold')

ax.set_ylabel('Número de donativos', fontsize=11, color='#555')
ax.set_title('Distribución de donativos por rango de monto\nevidencia de los cinco mundos del sector',
             fontsize=14, fontweight='bold', color='#222', pad=16)
ax.tick_params(axis='x', labelsize=10, rotation=20)
ax.tick_params(axis='y', labelsize=10, colors='#666')
ax.spines['bottom'].set_color('#ddd')
plt.tight_layout()
plt.savefig('g2_distribucion_rangos.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. Top sectores beneficiados ─────────────────────────────────────────────
top_sectores = (
    destino.groupby('Sector beneficiado').size()
    .sort_values(ascending=False)
    .head(8)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_sectores.index, top_sectores.values,
               color=COLORS['verde'], height=0.6)

for bar, val in zip(bars, top_sectores.values):
    ax.text(val + top_sectores.values.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}', va='center', fontsize=10, color=COLORS['verde'], fontweight='bold')

ax.set_xlabel('Número de donativos', fontsize=11, color='#555')
ax.set_title('Sectores beneficiados — top 8\n¿a quién llegan los donativos?',
             fontsize=14, fontweight='bold', color='#222', pad=16)

# Truncar etiquetas largas
labels_cortos = [l[:40] + '…' if len(l) > 40 else l for l in top_sectores.index]
ax.set_yticklabels(labels_cortos, fontsize=10)
ax.tick_params(axis='x', labelsize=10, colors='#666')
ax.spines['bottom'].set_color('#ddd')
plt.tight_layout()
plt.savefig('g3_sectores.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 4. Rubros de transparencia — solo top 5 ──────────────────────────────────
rubros_todos = transparencia['Rubro'].value_counts()

# Tomar top 5 y agrupar el resto en "Otros"
top5 = rubros_todos.head(5)
otros_val = rubros_todos.iloc[5:].sum()
rubros = pd.concat([top5, pd.Series({'Otros': otros_val})])

# Etiquetas cortas para la leyenda
def acortar(s, max_len=45):
    return s.replace('Organizaciones civiles y fideicomisos ', '').capitalize()

rubros.index = [acortar(i) if i != 'Otros' else 'Otros' for i in rubros.index]

colores_dona = [COLORS['azul'], COLORS['verde'], COLORS['ambar'],
                COLORS['morado'], COLORS['teal'], COLORS['gris']]

fig, ax = plt.subplots(figsize=(10, 6))
wedges, texts, autotexts = ax.pie(
    rubros.values,
    labels=None,
    autopct='%1.1f%%',
    colors=colores_dona,
    startangle=90,
    pctdistance=0.78,
    wedgeprops={'linewidth': 2, 'edgecolor': 'white'}
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_color('white')
    at.set_fontweight('bold')

ax.legend(wedges, rubros.index, loc='lower right', fontsize=10,
          frameon=False, bbox_to_anchor=(1.55, 0.05))

ax.set_title('Rubros declarados en transparencia\ndistribución del padrón de donatarias',
             fontsize=14, fontweight='bold', color='#222', pad=16)

centro = plt.Circle((0, 0), 0.5, color='white')
ax.add_patch(centro)
ax.text(0, 0, f'n={len(transparencia):,}', ha='center', va='center',
        fontsize=12, color='#444', fontweight='bold')

plt.tight_layout()
plt.savefig('g4_rubros.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Leer datos ────────────────────────────────────────────────────────────────
nomina = pd.read_excel(
    'InformeTransparencia2025.xlsx',
    sheet_name='Nómina'
)

# ── Estilo visual ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.grid': True,
    'grid.alpha': 0.12,
    'axes.axisbelow': True,
})

ROJO  = '#8B2F2F'
AMBAR = '#854F0B'

# ── Preparar datos ────────────────────────────────────────────────────────────
df = nomina.copy()

df['total_personas'] = (
    df['Voluntarios'] +
    df['Plantilla laboral']
)

df = df[
    (df['total_personas'] > 0) &
    (df['Monto salarios'] > 0)
].copy()

df['salario_por_persona'] = (
    df['Monto salarios'] /
    df['total_personas']
)

def acortar(s, n=45):
    return s[:n] + '…' if len(s) > n else s

# ── Caso 1: salario por persona > 1M ─────────────────────────────────────────
caso1 = (
    df[df['salario_por_persona'] > 1_000_000]
    .sort_values('salario_por_persona', ascending=False)
    .head(10)
    .copy()
)

caso1['label'] = caso1['Razón social'].apply(
    lambda x: acortar(x, 40)
)

# ── Caso 2: organizaciones con salario ~ $1 por empleado ────────────────────

df['ratio_1'] = (
    df['Monto salarios'] /
    df['Plantilla laboral']
)

caso2 = (
    df[
        (df['Plantilla laboral'] > 150_000) &
        (df['ratio_1'].between(0.95, 1.05))
    ]
    .sort_values('Plantilla laboral', ascending=False)
    .head(4)
    .copy()
)

caso2['label'] = caso2['Razón social'].apply(
    lambda x: acortar(x, 40)
)

# ── Figura ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))

fig.suptitle(
    'Anomalías en nómina — Informe de Transparencia\n'
    'Casos que requieren explicación',
    fontsize=16,
    fontweight='bold',
    color='#222',
    y=0.98
)

gs = GridSpec(2, 1, figure=fig, hspace=0.6)

# ── Panel 1: Salario por persona > 1M ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])

caso1_sorted = caso1.sort_values(
    'salario_por_persona',
    ascending=True
)

vals1 = caso1_sorted['salario_por_persona'].values / 1e6

bars1 = ax1.barh(
    caso1_sorted['label'],
    vals1,
    color=ROJO,
    height=0.6
)

for bar, val, (_, row) in zip(
    bars1,
    vals1,
    caso1_sorted.iterrows()
):

    ax1.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,

        f'${val:.1f}M/persona  '
        f'({int(row["Plantilla laboral"])} empleados + '
        f'{int(row["Voluntarios"])} voluntarios · '
        f'${row["Monto salarios"]/1e6:.1f}M total)',

        va='center',
        fontsize=9,
        color=ROJO
    )

ax1.set_xlabel(
    'Salario promedio por persona '
    '(plantilla + voluntarios, MXN millones)',
    fontsize=10,
    color='#555'
)

ax1.set_title(
    '① Organizaciones con costo promedio '
    'por persona mayor a $1,000,000\n'
    'Posible concentración de nómina en pocos puestos directivos',
    fontsize=11,
    fontweight='bold',
    color='#222',
    pad=10,
    loc='left'
)

ax1.tick_params(axis='y', labelsize=9)

ax1.set_xlim(0, vals1.max() * 1.55)

ax1.spines['bottom'].set_color('#ddd')

# ── Panel 2: $1 por empleado ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])

caso2_sorted = caso2.sort_values(
    'Plantilla laboral',
    ascending=True
)

bars2 = ax2.barh(
    caso2_sorted['label'],
    caso2_sorted['Plantilla laboral'] / 1e3,
    color=AMBAR,
    height=0.6
)

for bar, (_, row) in zip(
    bars2,
    caso2_sorted.iterrows()
):

    etiqueta = (
        f'{int(row["Plantilla laboral"]):,} empleados · '
        f'${row["Monto salarios"]:,.0f} total · '
        f'$1 por empleado'
    )

    ax2.text(
        bar.get_width() + 5,
        bar.get_y() + bar.get_height() / 2,
        etiqueta,
        va='center',
        fontsize=9,
        color=AMBAR
    )

ax2.set_xlabel(
    'Plantilla laboral (miles de empleados)',
    fontsize=10,
    color='#555'
)

ax2.set_title(
    '② Organizaciones con más de 700 mil empleados\n'
    'y salario equivalente a $1 por empleado',
    fontsize=11,
    fontweight='bold',
    color='#222',
    pad=10,
    loc='left'
)

ax2.tick_params(axis='y', labelsize=9)

ax2.set_xlim(
    0,
    (caso2_sorted['Plantilla laboral'] / 1e3).max() * 1.35
)

ax2.spines['bottom'].set_color('#ddd')

# ── Leyenda general ───────────────────────────────────────────────────────────
parches = [
    mpatches.Patch(
        color=ROJO,
        label='① Costo promedio por persona > $1M'
    ),

    mpatches.Patch(
        color=AMBAR,
        label='② Más de 700 mil empleados y $1 por empleado'
    ),
]

fig.legend(
    handles=parches,
    loc='lower center',
    ncol=2,
    fontsize=10,
    frameon=False,
    bbox_to_anchor=(0.5, 0.005)
)

# ── Guardar y mostrar ─────────────────────────────────────────────────────────
plt.savefig(
    'anomalias_nomina.png',
    dpi=150,
    bbox_inches='tight'
)

plt.show()

# ── Resumen consola ───────────────────────────────────────────────────────────
print('\n' + '=' * 80)
print('RESUMEN DE ANOMALÍAS PARA REUNIÓN CON CARLOS')
print('=' * 80)

print(
    f'\n① Costo/persona > $1M:   '
    f'{len(caso1)} organizaciones'
)

print(
    f'② $1 por empleado:       '
    f'{len(caso2)} organizaciones'
)

print(
    f'\nTotal único:  '
    f'{len(set(caso1.index) | set(caso2.index))} organizaciones'
)

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# ── Leer datos ────────────────────────────────────────────────────────────────
caratula = pd.read_excel("InformeTransparencia2024.xlsx", sheet_name="Carátula")
destino  = pd.read_excel("InformeTransparencia2024.xlsx", sheet_name="Destino de donativos")

# ── Merge: ligar cada donativo con el estado de origen de la donataria ────────
merged = destino.merge(
    caratula[["Folio", "Entidad federativa"]],
    on="Folio",
    suffixes=("_destino", "_origen")
)
merged = merged[merged["Monto"] > 0].copy()

def norm(s):
    return str(s).strip().title()

merged["origen"]    = merged["Entidad federativa_origen"].apply(norm)
merged["destino"]   = merged["Entidad federativa_destino"].apply(norm)
merged["municipio"] = merged["Municipio"].apply(norm)

# ── Filtrar top N por monto ───────────────────────────────────────────────────
TOP_ORIGINS = 8
TOP_DESTS   = 8
TOP_MUNS    = 12

top_orig = (merged.groupby("origen")["Monto"].sum()
            .sort_values(ascending=False).head(TOP_ORIGINS).index.tolist())
top_dest = (merged.groupby("destino")["Monto"].sum()
            .sort_values(ascending=False).head(TOP_DESTS).index.tolist())
top_mun  = (merged.groupby("municipio")["Monto"].sum()
            .sort_values(ascending=False).head(TOP_MUNS).index.tolist())

df = merged[
    merged["origen"].isin(top_orig) &
    merged["destino"].isin(top_dest) &
    merged["municipio"].isin(top_mun)
].copy()

# ── Agregaciones por nivel ────────────────────────────────────────────────────
link1 = df.groupby(["origen",  "destino"])["Monto"].sum().reset_index()   # L1→L2
link2 = df.groupby(["destino", "municipio"])["Monto"].sum().reset_index()  # L2→L3

# ── Construir lista de nodos ──────────────────────────────────────────────────
orig_nodes = sorted(link1["origen"].unique().tolist())
dest_nodes = sorted(link2["destino"].unique().tolist())
mun_nodes  = sorted(link2["municipio"].unique().tolist())

# Etiquetas con espacios para evitar colisiones entre niveles en Plotly
orig_labels = [n          for n in orig_nodes]
dest_labels = [n + " "   for n in dest_nodes]
mun_labels  = [n + "  "  for n in mun_nodes]
all_labels  = orig_labels + dest_labels + mun_labels

n_orig = len(orig_nodes)
n_dest = len(dest_nodes)

def idx_orig(name): return orig_nodes.index(name)
def idx_dest(name): return n_orig + dest_nodes.index(name)
def idx_mun(name):  return n_orig + n_dest + mun_nodes.index(name)

# ── Construir links ───────────────────────────────────────────────────────────
sources, targets, values = [], [], []

for _, row in link1.iterrows():
    if row["origen"] in orig_nodes and row["destino"] in dest_nodes:
        sources.append(idx_orig(row["origen"]))
        targets.append(idx_dest(row["destino"]))
        values.append(row["Monto"])

for _, row in link2.iterrows():
    if row["destino"] in dest_nodes and row["municipio"] in mun_nodes:
        sources.append(idx_dest(row["destino"]))
        targets.append(idx_mun(row["municipio"]))
        values.append(row["Monto"])

# ── Colores ───────────────────────────────────────────────────────────────────
BLUE  = "#185FA5"
GREEN = "#3B6D11"
AMBER = "#854F0B"

node_colors = [BLUE] * n_orig + [GREEN] * n_dest + [AMBER] * len(mun_nodes)

link_colors = [
    "rgba(24,95,165,0.22)" if s < n_orig else "rgba(59,109,17,0.22)"
    for s in sources
]

# ── Figura Sankey ─────────────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    orientation="h",
    arrangement="snap",
    node=dict(
        pad=18,
        thickness=22,
        line=dict(color="rgba(0,0,0,0)", width=0),
        label=all_labels,
        color=node_colors,
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
    )
))

fig.update_layout(
    title=dict(
        text="Flujo de donativos — México<br>"
             "<sup>Estado emisor → Estado receptor → Municipio | Monto en MXN</sup>",
        font=dict(size=18)
    ),
    font=dict(family="Arial", size=12),
    paper_bgcolor="#0d0f14",
    plot_bgcolor="#0d0f14",
    font_color="#c8c6c0",
    margin=dict(t=80, b=20, l=20, r=20),
    hoverlabel=dict(
        bgcolor="#1e2230",
        bordercolor="rgba(255,255,255,0.12)",
        font=dict(family="Courier New", size=12, color="#e8e6e0")
    )
)

fig.show()

# Opcional: exportar a HTML
# fig.write_html("sankey_donativos.html")

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FILE = "InformeTransparencia2024.xlsx"

# ── Leer datos ────────────────────────────────────────────────────────────────
caratula  = pd.read_excel(FILE, sheet_name="Carátula")
destino   = pd.read_excel(FILE, sheet_name="Destino de donativos")
generales = pd.read_excel(FILE, sheet_name="Generales")

# ── Auditoría de folios ───────────────────────────────────────────────────────
folios_caratula = set(caratula["Folio"].dropna().unique())
folios_destino  = set(destino["Folio"].dropna().unique())
coinciden       = folios_caratula & folios_destino
solo_caratula   = folios_caratula - folios_destino   # sin destino declarado

# ── Merge principal (destino conocido) ───────────────────────────────────────
merged = destino.merge(
    caratula[["Folio", "Entidad federativa"]],
    on="Folio",
    suffixes=("_destino", "_origen")
)
merged = merged[merged["Monto"] > 0].copy()

def norm(s):
    return str(s).strip().title()

merged["origen"]    = merged["Entidad federativa_origen"].apply(norm)
merged["destino"]   = merged["Entidad federativa_destino"].apply(norm)
merged["municipio"] = merged["Municipio"].apply(norm)

# ── Usar TODOS los registros ──────────────────────────────────────────────────
df = merged.copy()

link1 = df.groupby(["origen",  "destino"])["Monto"].sum().reset_index()
link2 = df.groupby(["destino", "municipio"])["Monto"].sum().reset_index()

# ── Monto desconocido por estado origen ──────────────────────────────────────
# Proxy: Total del estado de ingresos (Generales) para folios sin destino
sin_dest_caratula = caratula[caratula["Folio"].isin(solo_caratula)].copy()

gen_por_folio = (generales[["Folio", "Total del estado de ingresos"]]
                 .dropna()
                 .query("`Total del estado de ingresos` > 0"))

sin_dest_merged = sin_dest_caratula.merge(gen_por_folio, on="Folio", how="left")
sin_dest_merged["origen"] = sin_dest_merged["Entidad federativa"].apply(norm)

monto_desc_por_origen = (
    sin_dest_merged[sin_dest_merged["origen"].isin(top_orig)]
    .groupby("origen")["Total del estado de ingresos"]
    .sum()
    .fillna(0)
)

# ── Colores ───────────────────────────────────────────────────────────────────
BLUE  = "#185FA5"
GREEN = "#3B6D11"
AMBER = "#854F0B"
GREY  = "#73726C"

BLUE_LINK  = "rgba(24,95,165,0.22)"
GREEN_LINK = "rgba(59,109,17,0.22)"
GREY_LINK  = "rgba(120,110,100,0.35)"

# ── KPIs ──────────────────────────────────────────────────────────────────────
TOTAL_FOLIOS = len(folios_caratula)

FOLIOS_DESTINO = len(coinciden)

FOLIOS_SIN_DESTINO = len(solo_caratula)

MONTO_TOTAL = merged["Monto"].sum()

MONTO_DESCONOCIDO = (
    sin_dest_merged["Total del estado de ingresos"]
    .fillna(0)
    .sum()
)

# ── Figura ────────────────────────────────────────────────────────────────────
fig = go.Figure()

# ── Función Sankey ────────────────────────────────────────────────────────────
def sankey_trace(df_plot, visible=False):

    link1 = (
        df_plot.groupby(["origen", "destino"])["Monto"]
        .sum()
        .reset_index()
    )

    link2 = (
        df_plot.groupby(["destino", "municipio"])["Monto"]
        .sum()
        .reset_index()
    )

    # Municipios más importantes del filtro actual
    top_muns = (
        link2.groupby("municipio")["Monto"]
        .sum()
        .sort_values(ascending=False)
        .head(25)
        .index
    )

    link2 = link2[
        link2["municipio"].isin(top_muns)
    ]

    orig_nodes = sorted(df_plot["origen"].unique())
    dest_nodes = sorted(df_plot["destino"].unique())
    mun_nodes  = sorted(link2["municipio"].unique())

    unknown_node = "Destino desconocido"

    labels = (
        orig_nodes +
        [x + " " for x in dest_nodes] +
        [x + "  " for x in mun_nodes] +
        [unknown_node]
    )

    n_orig = len(orig_nodes)
    n_dest = len(dest_nodes)
    n_mun  = len(mun_nodes)

    idx_unknown = n_orig + n_dest + n_mun

    sources = []
    targets = []
    values  = []
    colors  = []

    # origen → destino
    for _, row in link1.iterrows():

        sources.append(
            orig_nodes.index(row["origen"])
        )

        targets.append(
            n_orig + dest_nodes.index(row["destino"])
        )

        values.append(row["Monto"])

        colors.append(BLUE_LINK)

    # destino → municipio
    for _, row in link2.iterrows():

        sources.append(
            n_orig + dest_nodes.index(row["destino"])
        )

        targets.append(
            n_orig + n_dest + mun_nodes.index(row["municipio"])
        )

        values.append(row["Monto"])

        colors.append(GREEN_LINK)

    # destino desconocido
    desconocidos = (
        sin_dest_merged[
            sin_dest_merged["origen"].isin(orig_nodes)
        ]
        .groupby("origen")["Total del estado de ingresos"]
        .sum()
    )

    for origen, monto in desconocidos.items():

        if monto > 0:

            sources.append(
                orig_nodes.index(origen)
            )

            targets.append(idx_unknown)

            values.append(monto)

            colors.append(GREY_LINK)

    node_colors = (
        [BLUE] * len(orig_nodes) +
        [GREEN] * len(dest_nodes) +
        [AMBER] * len(mun_nodes) +
        [GREY]
    )

    return go.Sankey(

        visible=visible,

        arrangement="snap",

        node=dict(
            pad=14,
            thickness=18,
            line=dict(color="rgba(0,0,0,0)", width=0),
            label=labels,
            color=node_colors,
        ),

        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors,
        )
    )

# ── Todos ─────────────────────────────────────────────────────────────────────
fig.add_trace(
    sankey_trace(df, visible=True)
)

# ── Filtros ───────────────────────────────────────────────────────────────────
estados_origen = sorted(df["origen"].unique())

for estado in estados_origen:

    fig.add_trace(
        sankey_trace(
            df[df["origen"] == estado]
        )
    )

# ── Dropdown origen ───────────────────────────────────────────────────────────
buttons_origen = []

buttons_origen.append(
    dict(
        label="Todos",
        method="update",
        args=[
            {"visible": [True] + [False]*len(estados_origen)},
            {"title": "Flujo de donativos — Todos"}
        ]
    )
)

for i, estado in enumerate(estados_origen):

    visible = [False]*(len(estados_origen)+1)

    visible[i+1] = True

    buttons_origen.append(
        dict(
            label=estado,
            method="update",
            args=[
                {"visible": visible},
                {"title": f"Flujo de donativos — {estado}"}
            ]
        )
    )

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(

    title=dict(
        text="Flujo de donativos en México",
        x=0.01,
        font=dict(size=20, color="#e8e6e0")
    ),

    paper_bgcolor="#0d0f14",
    plot_bgcolor="#0d0f14",

    font=dict(
        family="Arial",
        size=11,
        color="#d7d7d7"
    ),

    margin=dict(
        t=170,
        l=20,
        r=20,
        b=30
    ),

    updatemenus=[

        dict(
            buttons=buttons_origen,
            direction="down",
            x=0.01,
            y=1.12,
            bgcolor="#1c2230",
            bordercolor="#555",
            font=dict(color="white"),
            showactive=True
        )

    ]
)

# ── KPIs visuales ─────────────────────────────────────────────────────────────
kpis = [

    ("Folios", f"{TOTAL_FOLIOS:,}", 0.02),

    ("Con destino", f"{FOLIOS_DESTINO:,}", 0.20),

    ("Sin destino", f"{FOLIOS_SIN_DESTINO:,}", 0.38),

    ("Monto conocido", f"${MONTO_TOTAL/1e9:.2f}B", 0.60),

    ("Monto desconocido", f"${MONTO_DESCONOCIDO/1e9:.2f}B", 0.82),
]

for titulo, valor, x in kpis:

    fig.add_annotation(

        x=x,
        y=1.23,

        xref="paper",
        yref="paper",

        showarrow=False,

        align="center",

        text=(
            f"<b style='font-size:12px'>{titulo}</b><br>"
            f"<span style='font-size:20px;color:#4FA3FF'>{valor}</span>"
        ),

        bordercolor="rgba(255,255,255,0.12)",
        borderwidth=1,

        bgcolor="#151922",

        borderpad=8
    )

# ── Leyenda capas ─────────────────────────────────────────────────────────────
for x, txt, color in [

    (0.02, "● Estado origen", BLUE),

    (0.34, "● Estado destino", GREEN),

    (0.64, "● Municipios principales", AMBER),

    (0.90, "● Destino desconocido", GREY)

]:

    fig.add_annotation(

        x=x,
        y=1.05,

        xref="paper",
        yref="paper",

        showarrow=False,

        text=txt,

        font=dict(
            size=11,
            color=color
        )
    )

fig.show()

fig.write_html(
    "sankey_donativos_interactivo.html",
    include_plotlyjs="cdn"
)

